In [1]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR, AUX_COLS, ALPHAS
from src.data   import load_labels, load_coal_spectra
from src.submit import pack_submission
from src.model import get_cv_splits
from src.features import build_feature_matrix

In [2]:
label_map, aux_map = load_labels()
coal_type = COAL_TYPES[0]
train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)

y          = train_data['targets']
aux        = train_data['aux']
n_batches = train_data['n_batches']
groups  = train_data['groups']

splits = get_cv_splits(groups, n_batches)
X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True)

In [3]:
# Implement the RRR
from scipy.linalg import eigh

def fit_reduced_rank_ridge(X, Y, rank, alpha):
    """
    X: (n, p) standardized spectral features
    Y: (n, q) standardized targets [moisture, ash, H, S, CV]
    rank: int, number of latent factors r
    alpha: ridge regularization strength
    Returns coefficient matrix B (p, q) with rank <= r
    """
    n, p = X.shape
    q = Y.shape[1]

    # Step 1: Ridge regression solution (full rank) — this is your B_ridge
    XtX = X.T @ X + alpha * np.eye(p)
    XtY = X.T @ Y
    B_ridge = np.linalg.solve(XtX, XtY)  # (p, q), this is the "full rank" ridge coefficient

    # Step 2: Fitted values under full ridge
    Y_hat_full = X @ B_ridge  # (n, q)

    # Step 3: Reduced rank via generalized eigendecomposition
    # We want the top r eigenvectors of (Y_hat_full.T @ Y_hat_full) in the metric that
    # respects Y's covariance -- equivalent to SVD of Y_hat_full works for the simple case
    U, S, Vt = np.linalg.svd(Y_hat_full, full_matrices=False)
    V_r = Vt[:rank, :].T  # (q, r) -- top r right singular vectors

    # Step 4: Project B_ridge onto the rank-r output subspace
    B_rrr = B_ridge @ V_r @ V_r.T  # (p, q), rank <= r

    return B_rrr, B_ridge

In [10]:
y_full = np.column_stack([aux, y])
y_full.shape

(140, 5)

In [13]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()
Y_std = y_scaler.fit_transform(y_full)

In [ ]:
# Mean and std after normalization should be 0 and 1
np.isclose(np.mean(Y_std, axis=0),np.zeros(y_full.shape[1]))

array([ True,  True,  True,  True,  True])

In [17]:
np.isclose(np.std(Y_std, axis=0),np.ones(y_full.shape[1]))

array([ True,  True,  True,  True,  True])

In [ ]:
# Fit the RRR on the full dataset
B_rrr, B_full = fit_reduced_rank_ridge(X_spec, Y_std, rank=2, alpha=10.0)

In [36]:
# Get the prediciton and revert it back
Y_pred_std = X_spec @ B_rrr
Y_pred = y_scaler.inverse_transform(Y_pred_std)

CV_pred = Y_pred[:, -1]   # if CV is the last column
aux_pred = Y_pred[:, :4]  # moisture, ash, H, S

In [84]:
def print_rmse(y_pred, y_true):
    
    if len(y_pred.shape) == 1: # 1D prediction
        rmse = np.sqrt(np.mean((y_pred - y_true)**2))
        per_rmse = rmse / np.mean(y_true)
        print(f"RMSE for CV: {rmse.item(): .2f} | Percentage: {per_rmse*100: .2f}%")
    else:
        rmse_full = np.sqrt(np.mean((y_pred - y_true)**2, axis = 0))
        rmse_aux = rmse_full[:4]
        per_rmse_aux = rmse_aux / np.mean(aux, axis=0)

        for idx, name in enumerate(AUX_COLS):
            print(f"RMSE for {name}: {rmse_aux[idx]: .2f} | Percentage: {per_rmse_aux[idx]*100: .2f}%")

        if y_pred.shape[1] == 5:
            rmse_cv = rmse_full[-1]
            per_rmse_cv = rmse_cv / np.mean(y_true[:,-1])
            print(f"RMSE for CV: {rmse_cv.item(): .2f} | Percentage: {per_rmse_cv*100: .2f}%")

        rmse_std = rmse_full / np.std(y_true, axis=0)
        return rmse_std

In [64]:
# Get the oof predictions
oof = np.zeros_like(y_full)

for tr_idx, val_idx in splits:
    X_tr, y_tr = X_spec[tr_idx], Y_std[tr_idx]
    X_val = X_spec[val_idx]

    B_rrr, _ = fit_reduced_rank_ridge(X_tr, y_tr, rank=2, alpha=10.0)
    Y_pred_std = X_val @ B_rrr # (n_val, p) @ (p, q)
    Y_pred = y_scaler.inverse_transform(Y_pred_std)

    oof[val_idx] = Y_pred

In [85]:
_ = print_rmse(oof, y_full)

RMSE for 全水分:  1.24 | Percentage:  13.36%
RMSE for 灰分:  3.13 | Percentage:  14.35%
RMSE for 氢:  0.07 | Percentage:  3.05%
RMSE for 硫:  0.06 | Percentage:  12.63%
RMSE for CV:  321.23 | Percentage:  5.62%


In [41]:
# On your TRUE (ground truth) aux + CV matrix, standardized
U, S, Vt = np.linalg.svd(Y_std, full_matrices=False)
print(S)  # singular values
print(np.cumsum(S**2) / np.sum(S**2))  # cumulative variance explained

[23.41065551  9.78007767  7.36691574  1.07391755  0.93088285]
[0.78294113 0.91958387 0.99711451 0.99876208 1.        ]


Two remarks:
- The variance in the target matrix is explained by three latent variables
- It seems that the RRR approach does not significantly improve the prediction.

In [42]:
aux_scaler = StandardScaler()
aux_std = aux_scaler.fit_transform(aux)

U, S, Vt = np.linalg.svd(aux_std, full_matrices=False)
print(S)  # singular values
print(np.cumsum(S**2) / np.sum(S**2))  # cumulative variance explained

[20.31630907  9.71907553  7.18641239  1.06894026]
[0.73705788 0.90573722 0.99795958 1.        ]


In [108]:
def get_rrr_prediction(X, y, alpha = 10, weights = None):
    oof_aux = np.zeros_like(aux)

    # Handle the weighted case
    if weights is not None:
        y = y * weights

    for tr_idx, val_idx in splits:
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_val = X[val_idx]

        B_rrr, _ = fit_reduced_rank_ridge(X_tr, y_tr, rank=3, alpha=alpha)
        Y_pred_std = X_val @ B_rrr # (n_val, p) @ (p, q)

        if weights is not None:
            Y_pred_std = Y_pred_std / weights
            
        Y_pred = aux_scaler.inverse_transform(Y_pred_std)

        oof_aux[val_idx] = Y_pred

    return oof_aux

In [94]:
# Get the oof predictions
best_alpha = None
min_rmse_std = float('inf')

for alpha in ALPHAS:
    oof_aux = get_rrr_prediction(X_spec, aux_std, alpha=alpha)

    rmse_std = print_rmse(oof_aux, aux)
    mean_rmse_std = np.mean(rmse_std)
    print("---")
    print(f"Average RMSE Std for alpha = {alpha}: {mean_rmse_std: .2f}\n")

    if mean_rmse_std < min_rmse_std:
        best_alpha = alpha
        min_rmse_std = mean_rmse_std

print("=" * 30)
print(f"Best alpha: {best_alpha} | Min RMSE Std: {min_rmse_std}")

RMSE for 全水分:  1.26 | Percentage:  13.52%
RMSE for 灰分:  3.14 | Percentage:  14.43%
RMSE for 氢:  0.07 | Percentage:  3.08%
RMSE for 硫:  0.06 | Percentage:  12.71%
---
Average RMSE Std for alpha = 1.0:  0.94

RMSE for 全水分:  1.24 | Percentage:  13.31%
RMSE for 灰分:  3.14 | Percentage:  14.42%
RMSE for 氢:  0.07 | Percentage:  3.08%
RMSE for 硫:  0.05 | Percentage:  12.41%
---
Average RMSE Std for alpha = 10.0:  0.93

RMSE for 全水分:  1.23 | Percentage:  13.24%
RMSE for 灰分:  3.25 | Percentage:  14.93%
RMSE for 氢:  0.07 | Percentage:  3.20%
RMSE for 硫:  0.05 | Percentage:  12.10%
---
Average RMSE Std for alpha = 50.0:  0.94

RMSE for 全水分:  1.22 | Percentage:  13.13%
RMSE for 灰分:  3.34 | Percentage:  15.32%
RMSE for 氢:  0.07 | Percentage:  3.28%
RMSE for 硫:  0.05 | Percentage:  12.01%
---
Average RMSE Std for alpha = 100.0:  0.94

RMSE for 全水分:  1.19 | Percentage:  12.82%
RMSE for 灰分:  3.53 | Percentage:  16.18%
RMSE for 氢:  0.08 | Percentage:  3.46%
RMSE for 硫:  0.05 | Percentage:  12.04%
---
Av

In [116]:
# Get the best prediciton for alpha = 10
oof_aux = get_rrr_prediction(X_spec, aux_std)

In [113]:
# Stage 2
from sklearn.linear_model import RidgeCV

def stage2(oof_aux):
    oof_cv = np.zeros_like(y)

    for tr_idx, val_idx in splits:
        X_tr, y_tr = oof_aux[tr_idx], y[tr_idx]
        X_val = oof_aux[val_idx]
        stage2 = RidgeCV(alphas=ALPHAS)
        stage2.fit(X_tr, y_tr)
        oof_cv[val_idx] = stage2.predict(X_val)

    print("Prediction from the aux var alone:")
    print_rmse(oof_cv, y)

    # Stack with the X_spec matrix
    X_s2 = np.hstack((X_spec, oof_aux))
    scaler_s2 = StandardScaler()
    X_s2 = scaler_s2.fit_transform(np.nan_to_num(X_s2))

    oof_cv = np.zeros_like(y)

    for tr_idx, val_idx in splits:
        X_tr, y_tr = X_s2[tr_idx], y[tr_idx]
        X_val = X_s2[val_idx]
        stage2 = RidgeCV(alphas=ALPHAS)
        stage2.fit(X_tr, y_tr)
        oof_cv[val_idx] = stage2.predict(X_val)

    print("Prediction from the spec + aux var:")
    print_rmse(oof_cv, y)


In [117]:
stage2(oof_aux)

Prediction from the aux var alone:
RMSE for CV:  392.83 | Percentage:  6.88%
Prediction from the spec + aux var:
RMSE for CV:  253.57 | Percentage:  4.44%


In [ ]:
aux_scaler = StandardScaler()
aux_std = aux_scaler.fit_transform(aux)

oracle_model = RidgeCV(alphas=ALPHAS)
oracle_model.fit(aux_std, y)

for idx, (name, coef) in enumerate(zip(['moisture', 'ash', 'H', 'S'], oracle_model.coef_)):
    print(f"{name} | Coef: {coef:.2f} | Expected contribution: {rmse_std[idx] * coef: .2f}")
    

moisture | Coef: -45.48 | Expected contribution: -48.92
ash | Coef: -160.93 | Expected contribution: -135.36
H | Coef: 173.41 | Expected contribution:  144.88
S | Coef: 16.06 | Expected contribution:  15.38


Ash and H contribute the prediction error the most.

In [86]:
# oof_aux: your Stage-1 OOF predictions, shape (n, 4)
# Y_train[:, :4]: true aux values
resid = oof_aux - aux   # (n, 4) residual matrix

resid_corr = np.corrcoef(resid.T)
print(resid_corr)

# More precisely, reconstruct the *exact* expected CV error variance
# using the true error covariance instead of assuming independence:
c = oracle_model.coef_   # length-4 coefficient vector, on standardized aux scale
resid_std = resid / aux_scaler.scale_   # put residuals on same standardized scale as coef
Sigma = np.cov(resid_std.T)             # 4x4 covariance matrix of standardized residuals
predicted_CV_error_var = c @ Sigma @ c
print("Predicted CV RMSE from residual covariance:", np.sqrt(predicted_CV_error_var))

[[ 1.          0.54294296 -0.56948747  0.01589935]
 [ 0.54294296  1.         -0.98676079 -0.1944019 ]
 [-0.56948747 -0.98676079  1.          0.19818825]
 [ 0.01589935 -0.1944019   0.19818825  1.        ]]
Predicted CV RMSE from residual covariance: 313.4627757101035


The errors interact with each other.

## Coefficient corrected Stage 1 prediction

In [101]:
# Scale the axu_std to punish for error on important variables
weights = np.abs(oracle_model.coef_)
aux_weighted = aux_std * weights
np.std(aux_weighted, axis = 0)

array([ 45.482499  , 160.92553973, 173.41406065,  16.05629174])

In [109]:
# Get the oof predictions
best_alpha = None
min_rmse_std = float('inf')

for alpha in ALPHAS:
    oof_aux = get_rrr_prediction(X_spec, aux_std, alpha=alpha, weights= weights)

    rmse_std = print_rmse(oof_aux, aux)
    mean_rmse_std = np.mean(rmse_std)
    print("---")
    print(f"Average RMSE Std for alpha = {alpha}: {mean_rmse_std: .2f}\n")

    if mean_rmse_std < min_rmse_std:
        best_alpha = alpha
        min_rmse_std = mean_rmse_std

print("=" * 30)
print(f"Best alpha: {best_alpha} | Min RMSE Std: {min_rmse_std}")

RMSE for 全水分:  1.26 | Percentage:  13.59%
RMSE for 灰分:  3.19 | Percentage:  14.62%
RMSE for 氢:  0.07 | Percentage:  3.03%
RMSE for 硫:  0.05 | Percentage:  12.03%
---
Average RMSE Std for alpha = 1.0:  0.93

RMSE for 全水分:  1.24 | Percentage:  13.33%
RMSE for 灰分:  3.16 | Percentage:  14.50%
RMSE for 氢:  0.07 | Percentage:  3.07%
RMSE for 硫:  0.05 | Percentage:  11.75%
---
Average RMSE Std for alpha = 10.0:  0.91

RMSE for 全水分:  1.23 | Percentage:  13.25%
RMSE for 灰分:  3.26 | Percentage:  14.96%
RMSE for 氢:  0.07 | Percentage:  3.19%
RMSE for 硫:  0.05 | Percentage:  11.74%
---
Average RMSE Std for alpha = 50.0:  0.93

RMSE for 全水分:  1.22 | Percentage:  13.15%
RMSE for 灰分:  3.34 | Percentage:  15.35%
RMSE for 氢:  0.07 | Percentage:  3.28%
RMSE for 硫:  0.05 | Percentage:  11.79%
---
Average RMSE Std for alpha = 100.0:  0.94

RMSE for 全水分:  1.19 | Percentage:  12.82%
RMSE for 灰分:  3.53 | Percentage:  16.19%
RMSE for 氢:  0.08 | Percentage:  3.46%
RMSE for 硫:  0.05 | Percentage:  11.97%
---
Av

In [115]:
oof_aux = get_rrr_prediction(X_spec, aux_std, weights= weights)
stage2(oof_aux)

Prediction from the aux var alone:
RMSE for CV:  394.91 | Percentage:  6.91%
Prediction from the spec + aux var:
RMSE for CV:  244.49 | Percentage:  4.28%
